# FLAG 2027 - Notebook 03: Quick wins (challenge data only)

We test **6 versions** of the model, each adding **one** change to the previous one (an *ablation ladder* - this table goes straight into the paper):

| Version | Change | Why |
|---|---|---|
| A_official | official FOP settings | reference |
| B_fast_lr | learning rate 10x higher | Notebook 02 showed the model was still learning at epoch 198 |
| C_standardize | scale every feature to mean 0 / std 1 (statistics from TRAIN only) | VGGFace numbers have very different ranges |
| D_balanced | each batch = 16 people x 8 clips | some people have 226 clips, some 2 |
| E_crossmodal | extra loss that pulls a person's face and voice together directly + cosine scoring | FOP never directly trains face-vs-voice matching, yet that is what is scored |
| F_gender_hard | half the batches contain only one gender | the model must learn cues beyond gender (our 10-point gap!) |

Each version: **5 folds x 3 seeds = 15 trainings** on people never seen in training.

**How to run:** Runtime > Change runtime type > **T4 GPU**. Runtime > Run all. About 30-60 min.
Results are saved after every training, so if Colab disconnects just **Run all again** - it continues where it stopped.

**When done:** paste the **SUMMARY** box to Claude.

In [ ]:
# Cell 1 - Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - extract small csv/txt files + load training data (same as Notebook 02)
import os, json, time, zipfile, copy
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]

def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]

def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

Xf, yf = read_feat(find_one('train_English_faces.csv'), 4096)
Xv, yv = read_feat(find_one('train_English_voices.csv'), 192)
assert (yf == yv).all()
y = yf.astype(int)
lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}
XF = torch.tensor(Xf, device=dev)
XV = torch.tensor(Xv, device=dev)

SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
print('Loaded', len(y), 'clips,', len(ids), 'people')

In [ ]:
# Cell 3 - model, losses, metric
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)

class FOP(nn.Module):
    def __init__(self, n_class, d=128):
        super().__init__()
        self.face, self.voice = Branch(4096, d), Branch(192, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused), f, v

def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1)
    dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device)
    neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)

def crossmodal_loss(f, v, lab, tau=0.1):
    # supervised contrastive between modalities: each voice should pick out the faces of the same person
    sim = v @ f.t() / tau
    pos = (lab[:, None] == lab[None, :]).float()
    l_vf = -(pos * sim.log_softmax(1)).sum(1) / pos.sum(1)
    l_fv = -(pos.t() * sim.t().log_softmax(1)).sum(1) / pos.t().sum(1)
    return (l_vf.mean() + l_fv.mean()) / 2

def eer(labels, scores):
    # scores: HIGHER = more likely same person
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return (fpr[i] + fnr[i]) / 2 * 100

@torch.no_grad()
def same_score(model, xf, xv, scorer):
    # returns HIGHER = same person (we convert to the CodaBench direction only when writing files)
    model.eval()
    mf, sf, mv, sv = model.norm
    ft, vt, f, v = model.embed((xf - mf) / sf, (xv - mv) / sv)
    if scorer == 'l2tanh':
        return -(ft - vt).norm(dim=1).cpu().numpy()
    return (f * v).sum(1).cpu().numpy()   # cosine (f, v are unit length)

In [ ]:
# Cell 4 - honest validation (identical folds and pairs to Notebook 02 -> fair comparison)
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk = np.unique(y[idx])
    rows = {s: idx[y[idx] == s] for s in spk}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk)
        V.append(rng.choice(rows[s])); F.append(rng.choice(rows[s])); L.append(1)
        s = rng.choice(spk)
        cand = [t for t in spk if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk if t != s]
        t = rng.choice(cand)
        V.append(rng.choice(rows[s])); F.append(rng.choice(rows[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)

K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]
    rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)

FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]
    tr_idx = np.where([c not in val_ids for c in y])[0]
    vsets = [make_pairs(va_idx, same_gender_neg=False, seed=k),
             make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]
    FOLD_DATA.append((tr_idx, vsets))

def val_eers(model, vsets, scorer):
    return [eer(L, same_score(model, XF[F], XV[V], scorer)) for V, F, L in vsets]

In [ ]:
# Cell 5 - the 6 versions (each adds ONE change) + a general training function
A = dict(lr=1e-5, max_epochs=200, patience=20, std=False, sampler='random', xm=0.0, hard=0.0, scorer='l2tanh')
B = dict(A, lr=1e-4, max_epochs=100, patience=15)
C = dict(B, std=True)
D = dict(C, sampler='balanced')
E = dict(D, xm=1.0, scorer='cos')
Fv = dict(E, hard=0.5)
VARIANTS = {'A_official': A, 'B_fast_lr': B, 'C_standardize': C,
            'D_balanced': D, 'E_crossmodal': E, 'F_gender_hard': Fv}
SEEDS = [1, 2, 3]
BS, P_ID, K_CLIP, ALPHA = 128, 16, 8, 2.0

def balanced_batch(rng, pos_by_cls, cls_gender, hard):
    n = len(pos_by_cls)
    if rng.random() < hard:   # one-gender batch: all negatives share the gender
        gg = cls_gender[rng.integers(n)]
        pool = np.where(cls_gender == gg)[0]
    else:
        pool = np.arange(n)
    chosen = rng.choice(pool, size=min(P_ID, len(pool)), replace=False)
    return np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])

def train_model(cfg, tr_idx, vsets=None, seed=1, max_epochs=None):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    classes = np.unique(y[tr_idx])
    remap = {c: i for i, c in enumerate(classes)}
    ytr_np = np.array([remap[c] for c in y[tr_idx]])
    ytr = torch.tensor(ytr_np, device=dev)
    tr = torch.tensor(tr_idx, device=dev)
    if cfg['std']:   # statistics from the TRAINING rows only
        norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    else:
        z, o = torch.zeros(1, device=dev), torch.ones(1, device=dev)
        norm = (z, o, z, o)
    mf, sf, mv, sv = norm
    model = FOP(len(classes)).to(dev)
    model.norm = norm
    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=0.01)
    pos_by_cls = [np.where(ytr_np == k)[0] for k in range(len(classes))]
    cls_gender = np.array([id_gender[c] for c in classes])
    steps = len(tr_idx) // BS
    max_epochs = max_epochs or cfg['max_epochs']
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        if cfg['sampler'] == 'random':
            perm = torch.randperm(len(tr), device=dev)
            batches = [perm[b * BS:(b + 1) * BS] for b in range(steps)]
        else:
            batches = [torch.tensor(balanced_batch(rng, pos_by_cls, cls_gender, cfg['hard']), device=dev) for _ in range(steps)]
        for j in batches:
            fused, logits, f, v = model((XF[tr[j]] - mf) / sf, (XV[tr[j]] - mv) / sv)
            lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + ALPHA * opl_loss(fused, lab)
            if cfg['xm'] > 0:
                loss = loss + cfg['xm'] * crossmodal_loss(f, v, lab)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean(val_eers(model, vsets, cfg['scorer']))
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= cfg['patience']:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep

In [ ]:
# Cell 6 - run everything (RESUMABLE: finished trainings are skipped)
RESDIR = os.path.join(PROJECT, 'results'); os.makedirs(RESDIR, exist_ok=True)
CSV = os.path.join(RESDIR, 'nb03_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
total = len(VARIANTS) * len(SEEDS) * K
t0 = time.time()
for name, cfg in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep = train_model(cfg, tr_idx, vsets, seed=seed)
            e_std, e_gen = val_eers(model, vsets, cfg['scorer'])
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-14s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %d/%d done | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, len(done), total, (time.time() - t0) / 60))

In [ ]:
# Cell 7 - results table (the ablation ladder)
R = pd.read_csv(CSV)
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('variant').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                      overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab['gap'] = tab['gender'] - tab['std']
tab = tab.loc[[v for v in VARIANTS if v in tab.index]].round(2)
note('ABLATION LADDER (5-fold CV x 3 seeds; lower is better)')
note(tab.to_string())

BEST = tab['overall'].idxmin()
best_eps = R[R.variant == BEST]['best_ep']
FINAL_EPOCHS = int(np.median(best_eps))
note('BEST version: %s (overall %.2f) -> final model trains %d epochs' % (BEST, tab.loc[BEST, 'overall'], FINAL_EPOCHS))

In [ ]:
# Cell 8 - final model: best version on all 70 people, 3 seeds averaged -> submission ZIP
# Score direction for CodaBench: keep 'distance-like' (LOWER = same) until our first submission confirms it.
LOWER_IS_SAME = True

cfg = VARIANTS[BEST]
finals = [train_model(cfg, np.arange(len(y)), None, seed=s, max_epochs=FINAL_EPOCHS)[0] for s in SEEDS]

RUN = '03_' + BEST
OUT = '/content/sub_' + RUN
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang, fname in LANG_FILE.items():
        key = '/' + track + '/'
        pair_ids = [l.split()[0] for l in open(find_one(key, lang + '_test.txt')) if l.strip()]
        df_, _ = read_feat(find_one(key, lang + '_test_faces.csv'), 4096)
        dv_, _ = read_feat(find_one(key, lang + '_test_voices.csv'), 192)
        xf, xv = torch.tensor(df_, device=dev), torch.tensor(dv_, device=dev)
        s = np.mean([same_score(m, xf, xv, cfg['scorer']) for m in finals], axis=0)
        out = -s if LOWER_IS_SAME else s
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, out):
                f.write('%s %.6f\n' % (pid, v))

SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_PATH = os.path.join(SUBDIR, RUN + '.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
note('Submission ZIP saved: ' + ZIP_PATH[len(BASE):] + ' (LOWER_IS_SAME=%s)' % LOWER_IS_SAME)

In [ ]:
# Cell 9 - experiment log + summary
LOG = os.path.join(PROJECT, 'experiments.csv')
rows = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '03_' + v, 'change': 'ablation ladder step',
                      'seed': '1,2,3', 'cv_std': tab.loc[v, 'std'], 'cv_gender': tab.loc[v, 'gender'],
                      'cv_overall': tab.loc[v, 'overall'], 'codabench_dev': ''} for v in tab.index])
rows.to_csv(LOG, mode='a', header=not os.path.exists(LOG), index=False)

print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)